In [ ]:
import os
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import transforms

import medicalDataLoader_Enes as medicalDataLoader
from progressBar import printProgressBar
from utils_Enes import to_var, getTargetSegmentation, validate, show_predictions, DiceLoss, show_confusion
from UNet_Base_Enes import UNet

In [ ]:
FINAL = True
SEED = 0
if FINAL:
    MODEL_NAME = 'mean_teacher_final'
else:
    MODEL_NAME = 'mean_teacher'
SUPERVISED_PATH = '../baseline/models/final_dice/best.pth'
EPOCHS = 100
BATCH_SIZE = 8
LR = 0.001
AUGMENT = True
AUG_FLIP = False
AUG_SCALE = True
AUG_INTENSITY = True
WEIGHT_DECAY = 0
EMA_DECAY = 0.99
CONS_WEIGHT = 10.0
CONS_RAMPUP = 30
NOISE_STD = 0.03

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../../Data/'

In [ ]:
def perturb(x):
    contrast = torch.empty(x.size(0), 1, 1, 1, device=x.device).uniform_(0.8, 1.2)
    brightness = torch.empty(x.size(0), 1, 1, 1, device=x.device).uniform_(-0.1, 0.1)
    noise = torch.randn_like(x) * NOISE_STD
    return (x * contrast + brightness + noise).clamp(0, 1)


def cons_weight(epoch):
    if epoch >= CONS_RAMPUP:
        return CONS_WEIGHT
    p = 1 - epoch / CONS_RAMPUP
    return CONS_WEIGHT * np.exp(-5 * p * p)


@torch.no_grad()
def update_teacher(student, teacher, step):
    alpha = min(1 - 1 / (step + 1), EMA_DECAY)
    for t, s in zip(teacher.parameters(), student.parameters()):
        t.mul_(alpha).add_(s, alpha=1 - alpha)
    for t, s in zip(teacher.buffers(), student.buffers()):
        t.copy_(s)

In [ ]:
def runMeanTeacher(model_name):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    print('Dataset :', DATA_DIR)
    transform = transforms.Compose([transforms.ToTensor()])

    train_set = medicalDataLoader.MedicalImageDataset('train', DATA_DIR, transform=transform,
                                                      mask_transform=transform, augment=AUGMENT,
                                                      flip=AUG_FLIP, scale=AUG_SCALE, intensity=AUG_INTENSITY)
    if FINAL:
        val_train_set = medicalDataLoader.MedicalImageDataset('val', DATA_DIR, transform=transform,
                                                              mask_transform=transform, augment=AUGMENT,
                                                              flip=AUG_FLIP, scale=AUG_SCALE, intensity=AUG_INTENSITY)
        train_set = ConcatDataset([train_set, val_train_set])
        print('Mode final : train + val ({} images annotées)'.format(len(train_set)))
    train_eval_set = medicalDataLoader.MedicalImageDataset('train', DATA_DIR, transform=transform,
                                                           mask_transform=transform)
    val_set = medicalDataLoader.MedicalImageDataset('val', DATA_DIR, transform=transform,
                                                    mask_transform=transform)
    unlabeled_set = medicalDataLoader.UnlabeledDataset(DATA_DIR, transform=transform)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    train_eval_loader = DataLoader(train_eval_set, batch_size=BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
    unlabeled_loader = DataLoader(unlabeled_set, batch_size=BATCH_SIZE, shuffle=True)
    unlabeled_iter = iter(unlabeled_loader)

    student = UNet(num_classes=4)
    if torch.cuda.is_available():
        student.cuda()
    teacher = copy.deepcopy(student)
    for p in teacher.parameters():
        p.requires_grad = False
    print('Paramètres : {:,}'.format(sum(p.numel() for p in student.parameters())))

    CE_loss = nn.CrossEntropyLoss()
    dice_loss = DiceLoss()

    def loss_fn(out, target):
        return CE_loss(out, target) + dice_loss(out, target)

    optimizer = torch.optim.Adam(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    model_dir = os.path.join('models', model_name)
    os.makedirs(model_dir, exist_ok=True)
    best_path = os.path.join(model_dir, 'best.pth')
    history = {'train_loss': [], 'cons_loss': [], 'val_loss': [], 'train_dice': [],
               'student_dice': [], 'val_dice': [], 'val_hd': []}
    best_dsc, best_epoch = -1, 0
    step = 0

    for epoch in range(EPOCHS):
        student.train()
        teacher.train()
        w = cons_weight(epoch)
        losses, cons_losses = [], []
        for j, (images, labels, _) in enumerate(train_loader):
            images = to_var(images)
            target = getTargetSegmentation(to_var(labels))

            try:
                u_images, _ = next(unlabeled_iter)
            except StopIteration:
                unlabeled_iter = iter(unlabeled_loader)
                u_images, _ = next(unlabeled_iter)
            u_images = to_var(u_images)

            optimizer.zero_grad()
            sup_loss = loss_fn(student(images), target)
            student_probs = torch.softmax(student(perturb(u_images)), dim=1)
            with torch.no_grad():
                teacher_probs = torch.softmax(teacher(perturb(u_images)), dim=1)
            cons_loss = F.mse_loss(student_probs, teacher_probs)
            loss = sup_loss + w * cons_loss
            loss.backward()
            optimizer.step()
            update_teacher(student, teacher, step)
            step += 1

            losses.append(sup_loss.item())
            cons_losses.append(cons_loss.item())
            printProgressBar(j + 1, len(train_loader), prefix='[Training] Epoch: {} '.format(epoch),
                             length=15, suffix=' Loss: {:.4f}'.format(loss.item()))
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        _, train_dsc, _ = validate(teacher, train_eval_loader, loss_fn, with_hausdorff=False)
        _, student_dsc, _ = validate(student, val_loader, loss_fn, with_hausdorff=False)
        val_loss, val_dsc, val_hd = validate(teacher, val_loader, loss_fn)
        mean_dsc = np.mean(val_dsc)
        print('Epoch {:2d} | lr {:.6f} | loss train {:.4f}  cohérence {:.5f} (poids {:.2f})  val {:.4f}'.format(
            epoch, current_lr, np.mean(losses), np.mean(cons_losses), w, val_loss))
        print('   Train Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*train_dsc, np.mean(train_dsc)))
        print('   Val élève   VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*student_dsc, np.mean(student_dsc)))
        print('   Val prof    VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*val_dsc, mean_dsc))
        print('   Val   Hausdorff (px)  VD {:.1f}  Myo {:.1f}  VG {:.1f}'.format(*val_hd))

        history['train_loss'].append(np.mean(losses))
        history['cons_loss'].append(np.mean(cons_losses))
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dsc)
        history['student_dice'].append(student_dsc)
        history['val_dice'].append(val_dsc)
        history['val_hd'].append(val_hd)

        if mean_dsc > best_dsc:
            best_dsc, best_epoch = mean_dsc, epoch
            torch.save(teacher.state_dict(), best_path)
            print('   -> meilleur modèle enregistré')

    torch.save(teacher.state_dict(), os.path.join(model_dir, 'last.pth'))
    np.save(os.path.join(model_dir, 'history.npy'), history)
    print('Meilleur Dice moyen (prof) : {:.4f} (époque {})'.format(best_dsc, best_epoch))
    print('Dice moyen des 10 dernières époques : prof {:.4f}  élève {:.4f}'.format(
        np.mean([np.mean(d) for d in history['val_dice'][-10:]]),
        np.mean([np.mean(d) for d in history['student_dice'][-10:]])))

    if not FINAL:
        teacher.load_state_dict(torch.load(best_path))
    return teacher, val_loader

In [ ]:
net, val_loader = runMeanTeacher(MODEL_NAME)

if not FINAL:
    models = [('Mean Teacher', net)]
    if os.path.exists(SUPERVISED_PATH):
        supervised = UNet(num_classes=4)
        supervised.load_state_dict(torch.load(SUPERVISED_PATH, map_location='cpu'))
        if torch.cuda.is_available():
            supervised.cuda()
        models.insert(0, ('1A (supervisé)', supervised))

    CE_loss = nn.CrossEntropyLoss()
    for name, model in models:
        for post in [False, True]:
            _, dsc, hd = validate(model, val_loader, CE_loss, post_process=post)
            print('{} {} post-traitement'.format(name, 'avec' if post else 'sans'))
            print('   Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*dsc, np.mean(dsc)))
            print('   Hausdorff (px)  VD {:.1f}  Myo {:.1f}  VG {:.1f}'.format(*hd))

    show_predictions(net, val_loader, title='Validation Mean Teacher avec post-traitement', post_process=True)
    show_confusion(net, val_loader, title='(Mean Teacher, avec post-traitement)', post_process=True)